---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# LOWO with HistGradientBoosting + RandomizedSearchCV (GroupKFold)

**Objective:** Optimize hyperparameters using **GroupKFold** (cv=10) and validate with LOWO.

**Algorithm:** HistGradientBoostingRegressor

**Structure:**
1. Load data
2. Optimization with GroupKFold (cv=10)
3. LOWO validation
4. Statistical analyses
5. Visualizations
6. Save results to `results/histgb/`

---

## 1. Imports and Setup

In [ ]:
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import uniform, randint
from sonic_ml_utils import plot_well_profile_and_hexabin, statistics

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Directories ──────────────────────────────────────────────────────────────
RESULTS_DIR     = Path('../results/histgb')
METRICS_DIR     = RESULTS_DIR / 'metrics'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
MODELS_DIR      = RESULTS_DIR / 'models'
FIGURES_DIR     = RESULTS_DIR / 'figures'

for d in [METRICS_DIR, PREDICTIONS_DIR, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories set up.')
for d in [METRICS_DIR, PREDICTIONS_DIR, MODELS_DIR, FIGURES_DIR]:
    print(f'  {d}')

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/processed/wells_iqr_with_lithology.csv')

print(f'Data loaded')
print(f'Total samples : {len(df):,}')
print(f'Total wells   : {df["Well_Name"].nunique()}')
print(f'Columns       : {df.columns.tolist()}')

In [ ]:
# ── Global settings ──────────────────────────────────────────────────────────
FEATURES     = ['GR', 'RT90', 'RHOB', 'NPHI']
TARGET       = 'DT'
N_ITER       = 50
N_SPLITS     = 10
RANDOM_STATE = 42

wells = df['Well_Name'].unique()

print(f'Features      : {FEATURES}')
print(f'Target        : {TARGET}')
print(f'Wells         : {len(wells)}')
print(f'RandomSearch  : {N_ITER} iterations, GroupKFold cv={N_SPLITS}')

## 3. Hyperparameter Optimization (GroupKFold)

GroupKFold ensures that samples from the same well do not appear simultaneously in training and validation during tuning, preventing data leakage across wells.

In [ ]:
histgb_param_dist = {
    'max_iter'         : randint(100, 500),
    'max_depth'        : randint(4, 9),
    'learning_rate'    : uniform(0.02, 0.18),
    'max_leaf_nodes'   : randint(20, 50),
    'min_samples_leaf' : randint(10, 50),
    'l2_regularization': uniform(0, 10)
}

histgb_model = HistGradientBoostingRegressor(
    random_state=RANDOM_STATE,
    verbose=0
)

print(f'Search space: {len(histgb_param_dist)} hyperparameters, {N_ITER} iterations')

In [ ]:
# ── GroupKFold configuration validation ───────────────────────────────────────
print('GroupKFold CONFIGURATION VALIDATION')
print('=' * 60)

n_wells   = df['Well_Name'].nunique()
n_samples = len(df)

print(f'Dataset     : {n_samples:,} samples | {n_wells} wells')
print(f'N_SPLITS    : {N_SPLITS}')
print(f'Min required: N_SPLITS <= n_wells ({N_SPLITS} <= {n_wells})')

# ── Check 1: n_splits <= n_groups ────────────────────────────────────────────
if N_SPLITS > n_wells:
    print(f'\n[FAIL] N_SPLITS={N_SPLITS} > n_wells={n_wells}. '
          f'Reduce N_SPLITS to at most {n_wells}.')
    raise ValueError('N_SPLITS exceeds number of wells. Fix before proceeding.')
else:
    print(f'[OK]  N_SPLITS={N_SPLITS} <= n_wells={n_wells}')

# ── Check 2: inspect fold distribution ───────────────────────────────────────
X_val  = df[FEATURES].values
y_val  = df[TARGET].values
grps   = df['Well_Name'].values
gkf    = GroupKFold(n_splits=N_SPLITS)

print(f'\n{"Fold":>5s}  {"Train wells":>12s}  {"Val wells":>10s}  '
      f'{"Train samples":>14s}  {"Val samples":>12s}  Val well names')
print('-' * 110)

fold_stats = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_val, y_val, groups=grps), 1):
    train_wells = set(grps[train_idx])
    val_wells   = set(grps[val_idx])
    overlap     = train_wells & val_wells
    fold_stats.append({
        'fold'          : fold,
        'n_train_wells' : len(train_wells),
        'n_val_wells'   : len(val_wells),
        'n_train_samples': len(train_idx),
        'n_val_samples'  : len(val_idx),
        'overlap'        : len(overlap),
    })
    val_names = ', '.join(sorted(val_wells))
    print(f'{fold:>5d}  {len(train_wells):>12d}  {len(val_wells):>10d}  '
          f'{len(train_idx):>14,}  {len(val_idx):>12,}  {val_names}')

# ── Check 3: no overlap between train and validation wells ───────────────────
total_overlaps = sum(s['overlap'] for s in fold_stats)
print()
if total_overlaps > 0:
    print(f'[FAIL] {total_overlaps} well(s) appear in both train and validation '
          f'in at least one fold. GroupKFold is not working as expected.')
else:
    print(f'[OK]  No well appears in both train and validation in any fold.')

# ── Check 4: minimum wells per fold ──────────────────────────────────────────
min_val_wells = min(s['n_val_wells'] for s in fold_stats)
if min_val_wells < 1:
    print(f'[FAIL] At least one fold has 0 validation wells.')
else:
    print(f'[OK]  All folds have at least {min_val_wells} validation well(s).')

print()
print(f'Summary: {N_SPLITS} folds | '
      f'avg train wells={sum(s["n_train_wells"] for s in fold_stats)/N_SPLITS:.1f} | '
      f'avg val wells={sum(s["n_val_wells"] for s in fold_stats)/N_SPLITS:.1f} | '
      f'avg train samples={sum(s["n_train_samples"] for s in fold_stats)/N_SPLITS:,.0f}')
print('=' * 60)
print('GroupKFold configuration is valid. Safe to proceed.')


In [ ]:
X_all  = df[FEATURES].values
y_all  = df[TARGET].values
groups = df['Well_Name'].values

random_search = RandomizedSearchCV(
    estimator=histgb_model,
    param_distributions=histgb_param_dist,
    n_iter=N_ITER,
    scoring='r2',
    cv=GroupKFold(n_splits=N_SPLITS),
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=2
)

print(f'Starting optimization with GroupKFold (cv={N_SPLITS})...')
start = time.time()
random_search.fit(X_all, y_all, groups=groups)
time_opt = time.time() - start

best_params   = random_search.best_params_
best_cv_score = random_search.best_score_

print(f'\nOptimization completed in {time_opt/60:.2f} min')
print(f'Best R² (GroupKFold): {best_cv_score:.4f}')
print('\nBest hyperparameters:')
for k, v in sorted(best_params.items()):
    print(f'  {k:25s}: {v}')

In [ ]:
# Save best parameters and cv_results
with open(MODELS_DIR / 'histgb_best_params.json', 'w') as f:
    json.dump({'best_params': best_params, 'best_cv_score': float(best_cv_score)}, f, indent=2)

cv_results_df = pd.DataFrame(random_search.cv_results_)
cv_results_df.to_csv(MODELS_DIR / 'cv_results.csv', index=False)

print(f'Parameters saved : {MODELS_DIR}/histgb_best_params.json')
print(f'CV results saved : {MODELS_DIR}/cv_results.csv')
print(f'  R² range       : {cv_results_df["mean_test_score"].max() - cv_results_df["mean_test_score"].min():.4f}')

In [ ]:
# ── Load parameters from cache (skip execution above if already exists) ──────
with open(MODELS_DIR / 'histgb_best_params.json', 'r') as f:
    config = json.load(f)
    best_params = config['best_params']

print('best_params loaded.')
for k, v in sorted(best_params.items()):
    print(f'  {k:25s}: {v}')

## 4. LOWO Validation

In [ ]:
results_histgb = []
predictions_histgb  = []
start_lowo       = time.time()

for idx, test_well in enumerate(wells, 1):
    start_poco = time.time()

    train_df = df[df['Well_Name'] != test_well]
    test_df  = df[df['Well_Name'] == test_well]

    X_train, y_train = train_df[FEATURES].values, train_df[TARGET].values
    X_test,  y_test  = test_df[FEATURES].values,  test_df[TARGET].values

    print(f'[{idx:2d}/{len(wells)}] {test_well:25s} | Train: {len(X_train):6,} | Test: {len(X_test):5,}', end=' ')

    model = HistGradientBoostingRegressor(**best_params, random_state=RANDOM_STATE, verbose=0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    time_poco = time.time() - start_poco

    print(f'| R²={r2:.4f} | RMSE={rmse:.2f} | {time_poco:.1f}s')

    results_histgb.append({
        'Well_Name'       : test_well,
        'R2'              : r2,
        'RMSE'            : rmse,
        'MAE'             : mae,
        'N_samples_train' : len(X_train),
        'N_samples_test'  : len(X_test),
        'Time_seconds'    : time_poco
    })

    df_pred             = test_df[['DEPTH', 'Well_Name']].copy()
    df_pred['DT_real']  = y_test
    df_pred['DT_pred']  = y_pred
    for feat in FEATURES:
        df_pred[feat] = test_df[feat].values
    predictions_histgb.append(df_pred)

time_lowo = time.time() - start_lowo
print(f'\nLOWO completed in {time_lowo/60:.2f} min')

results_df_histgb   = pd.DataFrame(results_histgb)
df_predictions_histgb = pd.concat(predictions_histgb, ignore_index=True)

# Save
results_df_histgb.to_csv(METRICS_DIR / 'lowo_histgb_results.csv', index=False)
df_predictions_histgb.to_csv(PREDICTIONS_DIR / 'lowo_histgb_predictions.csv', index=False)

print(f'Metrics saved     : {METRICS_DIR}/lowo_histgb_results.csv')
print(f'Predictions saved : {PREDICTIONS_DIR}/lowo_histgb_predictions.csv')

In [ ]:
# ── Load results from cache (skip execution above if already exists) ──────────
results_df_histgb   = pd.read_csv(METRICS_DIR / 'lowo_histgb_results.csv')
df_predictions_histgb = pd.read_csv(PREDICTIONS_DIR / 'lowo_histgb_predictions.csv')

print(f'Results loaded.')
print(f'  Metrics     : {results_df_histgb.shape}')
print(f'  Predictions : {df_predictions_histgb.shape}')

## 5. Statistical Analyses

In [ ]:
# General statistics
print('=' * 80)
print('GENERAL STATISTICS — HistGradientBoosting LOWO')
print('=' * 80)
statistics.print_summary_table(results_df_histgb)

In [ ]:
# Full LOWO analysis
lowo_analysis = statistics.analyze_lowo_results(
    results_df=results_df_histgb,
    predictions_df=df_predictions_histgb,
    algorithm_name='HistGradientBoosting'
)

In [ ]:
# Problematic wells
R2_THRESHOLD = 0.70
problematic  = results_df_histgb[results_df_histgb['R2'] < R2_THRESHOLD]
good         = results_df_histgb[results_df_histgb['R2'] >= R2_THRESHOLD]

print(f'Good wells (R² >= {R2_THRESHOLD})         : {len(good)} ({len(good)/len(results_df_histgb)*100:.1f}%) | Mean R²: {good["R2"].mean():.4f}')
print(f'Problematic wells (R² < {R2_THRESHOLD})   : {len(problematic)} ({len(problematic)/len(results_df_histgb)*100:.1f}%)')
if len(problematic) > 0:
    for _, row in problematic.sort_values('R2').iterrows():
        print(f'  {row["Well_Name"]:28s} R²={row["R2"]:.4f} | RMSE={row["RMSE"]:.2f} | MAE={row["MAE"]:.2f}')

In [ ]:
# Residual analysis
y_true_all = df_predictions_histgb['DT_real'].values
y_pred_all = df_predictions_histgb['DT_pred'].values

residual_analysis = statistics.residual_analysis(y_true_all, y_pred_all)

print('RESIDUAL ANALYSIS')
print(f'  Mean       : {residual_analysis["mean"]:.4f} µs/ft')
print(f'  Median     : {residual_analysis["median"]:.4f} µs/ft')
print(f'  Std        : {residual_analysis["std"]:.4f} µs/ft')
print(f'  Skewness   : {residual_analysis["skewness"]:.4f}')
print(f'  Kurtosis   : {residual_analysis["kurtosis"]:.4f}')
print(f'  Normality  : {residual_analysis["normality"]}')
print(f'  Heteroscedasticity (corr): {residual_analysis["heteroscedasticity_corr"]:.4f}')

In [ ]:
# Error by DT range
error_by_bins = statistics.error_by_range(y_true=y_true_all, y_pred=y_pred_all, n_bins=10)
print('ERROR BY DT RANGE')
print(error_by_bins.to_string(index=False))

In [ ]:
# Confidence intervals
for metric, scores in [('R²',   results_df_histgb['R2'].values),
                        ('RMSE', results_df_histgb['RMSE'].values),
                        ('MAE',  results_df_histgb['MAE'].values)]:
    ci = statistics.calculate_confidence_interval(scores, confidence=0.95)
    print(f'{metric:6s}: {ci["mean"]:.4f}  95% CI: [{ci["lower_bound"]:.4f}, {ci["upper_bound"]:.4f}]  ±{ci["margin_of_error"]:.4f}')

In [ ]:
# Bootstrap confidence interval
bootstrap_ci = statistics.bootstrap_confidence_interval(
    y_true=y_true_all,
    y_pred=y_pred_all,
    metric_func=r2_score,
    n_bootstrap=1000,
    confidence=0.95
)

print(f'Bootstrap R² (1000 iterations):')
print(f'  Mean    : {bootstrap_ci["mean"]:.4f}')
print(f'  95% CI  : [{bootstrap_ci["lower_bound"]:.4f}, {bootstrap_ci["upper_bound"]:.4f}]')
print(f'  Width   : {bootstrap_ci["upper_bound"] - bootstrap_ci["lower_bound"]:.4f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(bootstrap_ci['values'], bins=40, color='steelblue', alpha=0.75, edgecolor='white')
ax.axvline(bootstrap_ci['mean'],        color='black',  lw=2,   linestyle='-',
           label=f'Mean: {bootstrap_ci["mean"]:.4f}')
ax.axvline(bootstrap_ci['lower_bound'], color='tomato', lw=1.5, linestyle='--',
           label=f'95% CI: [{bootstrap_ci["lower_bound"]:.4f}, {bootstrap_ci["upper_bound"]:.4f}]')
ax.axvline(bootstrap_ci['upper_bound'], color='tomato', lw=1.5, linestyle='--')
ax.set_xlabel('R²', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Bootstrap Distribution of R² — HistGradientBoosting LOWO (1000 iterations)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'lowo_histgb_bootstrap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
final_model = HistGradientBoostingRegressor(**best_params, random_state=RANDOM_STATE, verbose=0)
final_model.fit(df[FEATURES].values, df[TARGET].values)

importance_df = statistics.feature_importance_analysis(
    model=final_model,
    feature_names=FEATURES,
    X=df[FEATURES].values,  # ← required for HistGB
    y=df[TARGET].values      # ← required for HistGB
)

print('FEATURE IMPORTANCE')
print(importance_df.to_string(index=False))

## 6. Visualizations

In [ ]:
# Profile + scatter: all wells
plot_well_profile_and_hexabin(
    predictions_df=df_predictions_histgb,
    well_name=None,
    figsize=(10, 8),
    cmap='YlGnBu',
    gridsize=70,
    save_path=str(FIGURES_DIR / 'lowo_histgb_well.png')
)

In [ ]:
# Top 5 best and worst wells
best_5  = results_df_histgb.nlargest(5, 'R2')
worst_5 = results_df_histgb.nsmallest(5, 'R2')

print('TOP 5 BEST:')
for _, row in best_5.iterrows():
    print(f'  {row["Well_Name"]:28s} R²={row["R2"]:.4f}')

print('\nTOP 5 WORST:')
for _, row in worst_5.iterrows():
    print(f'  {row["Well_Name"]:28s} R²={row["R2"]:.4f}')
